Préparer les données

In [96]:


with open("schekspear_dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Nombre de caractères :", len(text))



Nombre de caractères : 1115394


tokanisation -> on va adopter la tokanisation par mot car on peut traite chaque 3 mot consecutif pour calculer la proba

In [97]:
import nltk
from nltk import word_tokenize,sent_tokenize
i = 0
sentences_sep = []
sentences = sent_tokenize(text)
for sentence in sentences : 
    sen = "START " + sentence + " END"
    sentences_sep.append(sen)

tokenized_sentences = []
for sentence in sentences_sep:
    # Tokeniser la phrase en mots
    tokens = word_tokenize(sentence, language="french")
    tokenized_sentences.append(tokens)

print(tokenized_sentences[:20])
#maintenat les tokens sont pret 

[['START', 'First', 'Citizen', ':', 'Before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'END'], ['START', 'All', ':', 'Speak', ',', 'speak', '.', 'END'], ['START', 'First', 'Citizen', ':', 'You', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', '?', 'END'], ['START', 'All', ':', 'Resolved', '.', 'END'], ['START', 'resolved', '.', 'END'], ['START', 'First', 'Citizen', ':', 'First', ',', 'you', 'know', 'Caius', 'Marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', '.', 'END'], ['START', 'All', ':', 'We', "know't", ',', 'we', "know't", '.', 'END'], ['START', 'First', 'Citizen', ':', 'Let', 'us', 'kill', 'him', ',', 'and', 'we', "'ll", 'have', 'corn', 'at', 'our', 'own', 'price', '.', 'END'], ['START', "Is't", 'a', 'verdict', '?', 'END'], ['START', 'All', ':', 'No', 'more', 'talking', 'o', "n't", ';', 'let', 'it', 'be', 'done', ':', 'away', ',', 'away', '!', 'END'], ['START', 'Second', 'Citizen', ':', 'One', 'word', ',', 'good', 'citize

puisque les tokens sont pret avec leur seperateur on va passer a l etape suivante --> former les trigrames

In [98]:
tigrams = []
for tokens in tokenized_sentences : 
    for i in range(len(tokens) -2 ): 
        tigram= (tokens[i],tokens[i+1],tokens[i+2])
        tigrams.append(tigram)

print(tigrams[:50])

[('START', 'First', 'Citizen'), ('First', 'Citizen', ':'), ('Citizen', ':', 'Before'), (':', 'Before', 'we'), ('Before', 'we', 'proceed'), ('we', 'proceed', 'any'), ('proceed', 'any', 'further'), ('any', 'further', ','), ('further', ',', 'hear'), (',', 'hear', 'me'), ('hear', 'me', 'speak'), ('me', 'speak', '.'), ('speak', '.', 'END'), ('START', 'All', ':'), ('All', ':', 'Speak'), (':', 'Speak', ','), ('Speak', ',', 'speak'), (',', 'speak', '.'), ('speak', '.', 'END'), ('START', 'First', 'Citizen'), ('First', 'Citizen', ':'), ('Citizen', ':', 'You'), (':', 'You', 'are'), ('You', 'are', 'all'), ('are', 'all', 'resolved'), ('all', 'resolved', 'rather'), ('resolved', 'rather', 'to'), ('rather', 'to', 'die'), ('to', 'die', 'than'), ('die', 'than', 'to'), ('than', 'to', 'famish'), ('to', 'famish', '?'), ('famish', '?', 'END'), ('START', 'All', ':'), ('All', ':', 'Resolved'), (':', 'Resolved', '.'), ('Resolved', '.', 'END'), ('START', 'resolved', '.'), ('resolved', '.', 'END'), ('START', 'Fi

apres avoir creer les tigrammes on va maintenat faire compter la frequence de chaque tigramme dans tout le txt

In [99]:
from collections import defaultdict

tigrams_count = defaultdict(int)
tigrams_prefix_count=defaultdict(int)

for mot1,mot2,mot3 in biagrams :
    tigrams_count[(mot1,mot2,mot3)] +=1 
    tigrams_prefix_count[(mot1, mot2)] += 1


sorted_tigrams = sorted(tigrams_count.items(), key=lambda x: x[1], reverse=True)
print("Top 12 tigrammes :")
for tigram, count in sorted_tigrams[:20]:
    print(f"{tigram}: {count}")

Top 12 tigrammes :
('START', 'KING', 'RICHARD'): 229
(',', 'sir', ','): 218
('START', 'GLOUCESTER', ':'): 215
(',', 'my', 'lord'): 194
('DUKE', 'VINCENTIO', ':'): 193
('START', 'DUKE', 'VINCENTIO'): 187
('START', 'ROMEO', ':'): 160
('START', 'MENENIUS', ':'): 156
('him', '.', 'END'): 154
('START', 'PETRUCHIO', ':'): 153
('it', '.', 'END'): 146
('START', 'CORIOLANUS', ':'): 145
(':', 'O', ','): 144
(':', 'Why', ','): 143
(':', 'Ay', ','): 142
('KING', 'RICHARD', 'III'): 138
('RICHARD', 'III', ':'): 138
(',', 'I', "'ll"): 135
('OF', 'YORK', ':'): 130
('me', '.', 'END'): 127


In [100]:
tigrams_proba = defaultdict(int)
for (mot1,mot2,mot3),count in tigrams_count.items() :
    tigrams_proba[(mot1,mot2,mot3)] = count/tigrams_prefix_count[(mot1,mot2)]

print("\nProbabilités de quelques tigrammes :")
for bigram, prob in list(tigrams_proba.items())[:30]:
    print(f"P({bigram[2]} |  {bigram[0]} | {tigram[1]}) = {prob:.4f}")


Probabilités de quelques tigrammes :
P(Citizen |  START | .) = 0.1737
P(: |  First | .) = 1.0000
P(Before |  Citizen | .) = 0.0204
P(we |  : | .) = 0.2500
P(proceed |  Before | .) = 1.0000
P(any |  we | .) = 0.5000
P(further |  proceed | .) = 1.0000
P(, |  any | .) = 0.3333
P(hear |  further | .) = 0.2500
P(me |  , | .) = 0.7059
P(speak |  hear | .) = 0.3529
P(. |  me | .) = 0.5714
P(END |  speak | .) = 1.0000
P(: |  START | .) = 0.4615
P(Speak |  All | .) = 0.0526
P(, |  : | .) = 0.1818
P(speak |  Speak | .) = 0.3333
P(. |  , | .) = 0.2778
P(You |  Citizen | .) = 0.0714
P(are |  : | .) = 0.1790
P(all |  You | .) = 0.0208
P(resolved |  are | .) = 0.0714
P(rather |  all | .) = 0.5000
P(to |  resolved | .) = 1.0000
P(die |  rather | .) = 0.5000
P(than |  to | .) = 0.0400
P(to |  die | .) = 1.0000
P(famish |  than | .) = 0.1667
P(? |  to | .) = 0.3333
P(END |  famish | .) = 1.0000


ajouter le lissage pour gerer les noms qui n existe pas dans cette dataset 

In [101]:
vocab = set()
for mot1, mot2,mot3 in tigrams_count.keys():
    vocab.add(mot1)
    vocab.add(mot2)
    vocab.add(mot3)
vocab_size = len(vocab)
vocab_size

# Appliquer le lissage de Laplace
# Conditionné sur le contexte (mot1, mot2)
smoothed_probs = {}
for (mot1, mot2, mot3) in tigrams_count:
    context_count = tigrams_prefix_count[(mot1, mot2)]
    smoothed_probs[(mot1, mot2, mot3)] = (tigrams_count[(mot1, mot2, mot3)] + 1) / (context_count)




pour faciliter la prediction de mot qui suit on va adopter la structure imbrique de dictionaire

In [102]:
transition_probs = defaultdict(dict)
for (mot1, mot2, mot3), prob in smoothed_probs.items():
    transition_probs[(mot1, mot2)][mot3] = prob

for mot3, prob in transition_probs[('I','love')].items():
        print(f"  P({mot3} | I | want ) = {prob:.6f}")

  P(them | I | want ) = 0.062500
  P(him | I | want ) = 0.156250
  P(Hastings | I | want ) = 0.062500
  P(thy | I | want ) = 0.125000
  P(myself | I | want ) = 0.125000
  P(. | I | want ) = 0.093750
  P(; | I | want ) = 0.062500
  P(now | I | want ) = 0.062500
  P(thee | I | want ) = 0.125000
  P(the | I | want ) = 0.187500
  P(you | I | want ) = 0.062500
  P(a | I | want ) = 0.093750
  P(her | I | want ) = 0.093750
  P(you. | I | want ) = 0.062500
  P(Lucentio | I | want ) = 0.062500
  P(no | I | want ) = 0.062500


maintenat on va generer

In [103]:
import random

def generate_sentence(transition_probs, max_length=20):
    current_word = "we " + "are "
    sentence = [current_word]

    for _ in range(max_length):
        # Récupérer les mots possibles et leurs probabilités
        next_words = transition_probs.get(("We","are"), {})
        if not next_words:
            break  # Pas de transition possible

        # Choisir le prochain mot aléatoirement selon les probabilités
        next_word = random.choices(
            list(next_words.keys()),
            weights=list(next_words.values()),
            k=1
        )[0]

        sentence.append(next_word)
        current_word = next_word

        # Arrêter si on atteint END
        if current_word == "END":
            break

    # Retirer START et END pour afficher la phrase
    sentence = [word for word in sentence if word not in ["START", "END"]]
    return " ".join(sentence)

# Générer 5 phrases
for _ in range(10):
    print(generate_sentence(transition_probs))

we are  lucky but undone all not to blest all to fit to to not accounted tougher blest undone blest all all
we are  true yours true true less merely convented yours advertised merely not less inforced inforced true merely advertised all fit the
we are  convented merely to merely less but convented merely less yours less accounted advertised accounted all undone , blest true all
we are  but less , not undone undone not to but lucky the accounted fit but , less merely merely not inforced
we are  amazed fit amazed merely tougher amazed yours but yours to accounted lucky to accounted the , lucky convented to ,
we are  blest but fit not the the amazed the , not amazed not accounted to lucky tougher undone fit , undone
we are  the all less all to inforced not , not inforced accounted the all all convented tougher amazed fit not inforced
we are  lucky lucky convented fit to undone all convented the true not , yours fit all accounted but merely amazed all
we are  advertised lucky convented the

# ======================
# 2. ÉVALUATION SUR UN TEXTE DE TEST
# ======================


In [104]:
import math
test_text = "vice in him. You must in no way say he is covetous."
test_tokens = word_tokenize(test_text)
test_trigrams = [
    (test_tokens[i], test_tokens[i+1], test_tokens[i+2])
    for i in range(len(test_tokens) - 2)
]

log_probs = []
for mot1, mot2, mot3 in test_trigrams:
    context_count = tigrams_prefix_count.get((mot1, mot2), 0)
    # Probabilité OOV avec Laplace
    prob = transition_probs[(mot1, mot2)].get(
        mot3,
        1 / (context_count + vocab_size + 1)  
    )
    log_probs.append(math.log2(prob))

avg_log_prob = sum(log_probs) / (len(log_probs) + 0.1)
perplexity = 2 ** (-avg_log_prob)
print(f"✅ Perplexité corrigée : {perplexity:.2f}")

✅ Perplexité corrigée : 8.21
